In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Reading the dataset into a pandas DataFrame so we can analyze it.

data = pd.read_csv('youtube_ad_revenue_dataset.csv')


In [ ]:
data

In [ ]:
# checking the max ad revenue and max subscribers

max_value_ad = data['ad_revenue_usd'].max()
print(max_value_ad)
mx_v= data['subscribers'].max()
print(mx_v)

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [ ]:
# Checking the size of the data, the types, and looking for any missing values.

print("Shape:", data.shape)
print("\nData Types:")
print(data.dtypes)

print("\nMissing Values (%):")
print((data.isnull().mean() * 100).sort_values(ascending=False))

print("\nDuplicate Rows:", data.duplicated().sum())

In [ ]:
data.describe()

In [ ]:
print("Is video_id unique?", data['video_id'].is_unique)
print("Unique videos:", data['video_id'].nunique())

#### Data type fix and time features

In [ ]:
data['date'] = pd.to_datetime(data['date'])

# Extracting the date 
data['year'] = data['date'].dt.year
data['month'] = data['date'].dt.month
data['dayofweek'] = data['date'].dt.dayofweek


#### Logical transformation

In [ ]:
# DATA CLEANING - LOGICAL
# Finding rows with impossible values (like negative watch time or revenue).
# Note: watch_time_minutes is the TOTAL watch time across all viewers for that video.
# So watch_time_minutes / views = avg watch time per view, which must be <= video_length_minutes.

logical_errors = (
    (data['views'] < 0) |
    (data['likes'] < 0) |
    (data['comments'] < 0) |
    (data['watch_time_minutes'] < 0) |
    (data['video_length_minutes'] < 0) |
    (data['ad_revenue_usd'] < 0) |
    # Avg watch time per view cannot exceed the video length
    ((data['watch_time_minutes'] / data['views'].clip(lower=1)) > data['video_length_minutes'])
)

print("Logical error rows:", logical_errors.sum())


In [ ]:
# DATA CLEANING = REMOVING ERRORS 
# Dropping the impossible rows we found above.

data = data[~logical_errors].copy()

In [ ]:
# Checking skewness of numeric columns before imputation
print("Skewness of numeric columns:")
print(data[['likes', 'comments', 'watch_time_minutes']].skew())


#### Missing values

In [ ]:
# histogram 
data['likes'].hist(bins=50)
plt.title("Distribution of Likes (Before Imputation)")
plt.xlabel("Likes")
plt.ylabel("Frequency")
plt.show()


In [ ]:
# Filling missing numeric data with medians (to ignore extreme outliers)
# and filling missing text categories with 'Unknown'.

# Separate columns
numeric_cols = data.select_dtypes(include=np.number).columns
categorical_cols = data.select_dtypes(include='object').columns

# I checked the skeweness however the value is almost zero,
# so we can use both mean and median (I am going to use median because its a universal practice )
for col in numeric_cols:
    if data[col].isnull().sum() > 0:
         data[col]=  data[col].fillna(data[col].median())

# Mode / 'Unknown' for categorical(there is no missing values however I put it here if in future this will be my plug and play)
for col in categorical_cols:
    if data[col].isnull().sum() > 0:
        data[col]= data[col].fillna("Unknown")


#### Log Tranformations


In [ ]:
# Here the data is perfectly balanced however in the real world scenario the data will be heavily skewed 
# so as per the industry norms I am doing log tranformations so our machine learning models perform fairly.

data['log_views'] = np.log1p(data['views'])
data['log_subscribers'] = np.log1p(data['subscribers'])
data['log_ad_revenue_usd'] = np.log1p(data['ad_revenue_usd'])

#### Log Transformation Rationale

Log transformation applied to:
1. Reduce scale dominance (views & revenue vary by orders of magnitude)
2. Improve numerical stability for linear models
3. Align with multiplicative monetization effects

In [ ]:
print("Duplicate Rows:", data.duplicated().sum())

In [ ]:
data = data.drop_duplicates()

In [ ]:
# Checking the size of the data, the types, and looking for any missing values.

print("Final shape:", data.shape)
print("Remaining missing values:", data.isnull().sum().sum())
print("Duplicates:", data.duplicated().sum())

# EDA (Exploratory Data Analysis)

### Univariate Analysis

In [ ]:
# this was performed earlier in the notebook before the log transformations(howver its a part of EDA so including it here)
numeric_univariate_cols = [
    'views',
    'likes',
    'comments',
    'watch_time_minutes',
    'video_length_minutes',
    'subscribers',
    'ad_revenue_usd'
]


In [ ]:
# Visualizing the distribution of the data to find insights.
# histplot
for col in numeric_univariate_cols:
    plt.figure(figsize=(5,3))
    sns.histplot(data[col], bins=50, kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()


In [ ]:
# Visualizing the distribution of the data to find insights.

plt.figure(figsize=(6,3))
sns.countplot(
    y='category',
    data=data,
    order=data['category'].value_counts().index
)
plt.title("Video Count by Category")
plt.show()

In [ ]:
# Visualizing the distribution of the data to find insights.

sns.countplot(x='device', data=data)
plt.title("Device Distribution")
plt.show()

In [ ]:
# Visualizing the distribution of the data to find insights.

top_countries = data['country'].value_counts().head(10)

sns.barplot(
    x=top_countries.index,
    y=top_countries.values
)
plt.xticks(rotation=45)
plt.title("Top 10 Countries by Video Count")
plt.show()


#### Univariate Analysis Summary

Categories, devices, and countries all have almost equal counts
Most numeric variables show synthetic or near-uniform distributions.

In [ ]:
# Numeric features exhibit synthetic, near-uniform distributions.

### Bi-Variate Analysis

In [ ]:
#    FEATURE ENGINEERING: ENGAGEMENT METRICS  
# Creating new features that measure the 'quality' of interaction (retention) instead of purely its scale.

# clip(lower=1) prevents division-by-zero before it happens,
# while replace(np.inf, 0) fixes the damage after bad math has already occurred.
views_safe = data['views'].clip(lower=1)

data['engagement_rate'] = (data['likes'] + data['comments']) / views_safe
data['watch_time_per_view'] = data['watch_time_minutes'] / views_safe


In [ ]:
sns.scatterplot(
    x='watch_time_per_view',
    y='ad_revenue_usd',
    data=data,
    alpha=0.3
)
plt.title("Watch Time per View vs Ad Revenue")
plt.show()


In [ ]:
#   EXPLORATORY DATA ANALYSIS (EDA)  
# Visualizing the distribution of the data to find insights.

top_countries = data['country'].value_counts().head(10).index

plt.figure(figsize=(5,3))
sns.boxplot(
    x='country',
    y='log_ad_revenue_usd',
    data=data[data['country'].isin(top_countries)]
)
plt.xticks(rotation=45)
plt.title("Log Ad Revenue by Country (Top 10)")
plt.show()


In [ ]:
# correlation heatmap 
corr_features = [
    'watch_time_per_view',
    'engagement_rate',
    'video_length_minutes',
    'log_ad_revenue_usd'
]
corr = data[corr_features].corr()
plt.figure(figsize=(5,3))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Matrix (Engineered Features)")
plt.show()

#### EDA Summary

## EDA Summary+

Watch time per view is the strongest factor affecting ad revenue.

Engagement rate has a weak effect when considered alone.

Category, device, and country change revenue distribution, but none dominate by themselves.

Raw metrics like views and watch time are redundant, so transformed features are more useful.

#### EDA Technical Interpretation

Viewer retention matters more than pure views for earning ad revenue.

Watch time per view replaces raw watch time to remove redundancy and reduce correlation issues.

Log transformations help stabilize relationships between features and revenue.

Categorical features add context, but most predictive power comes from retention and reach.

#### Key Insight

the watch_time_per_view has high impact on ad revenue(after log transform the relationship is stable) and is more than the log_views

In [ ]:
#    DROPPING REDUNDANT COLUMNS  
# Dropping columns like video_id and raw views since we engineered better features for our model.

# Dropping the redundant columns before modelling
target = 'log_ad_revenue_usd'
data_model = data.drop(columns=[
    'video_id',
    'date',
    'views',
    'likes',
    'comments',
    'watch_time_minutes',
    'ad_revenue_usd'  
])
data_model = data_model.reset_index(drop=True)

print( data_model.shape)
print(data_model.info())

In [ ]:
numeric_features = [
    'log_views',
    'log_subscribers',
    'watch_time_per_view',
    'engagement_rate',
    'video_length_minutes',
    'year',
    'month',
    'dayofweek'
]

categorical_features = [
    'category',
    'device',
    'country'
]


In [ ]:
#   DATA SPLITTING  
# Splitting the valid data into 80% for training the model,
# and 20% for testing its accuracy.

# train test split
from sklearn.model_selection import train_test_split

X = data_model.drop(columns=[target])
y = data_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
#    MODEL PIPELINE PREPARATION  
# Setting up standard scaling for numerical data and one-hot encoding for categorical text.

# (StandardScaler dono ki "unit" (cm, kg, rupees) chheen leta hai aur unhe ek common scale par le aata hai. 
# Wo sabko ek common benchmark se compare karta hai: "Average se kitna door?".)

# OneHotEncoder har category ke liye ek naya column bana deta hai. Wo sirf 0 aur 1 use karta hai. (ML are essentially
# mathematical equations so they dont undertsnad text so tht is y v convt categ columns to new clms using onehotencodr)


from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features), 
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)


In [ ]:
# MODEL ALGORITHMS
# Defining multiple machine learning algorithms.

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, SGDRegressor

models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.01),
    'ElasticNet': ElasticNet(alpha=0.01, l1_ratio=0.5),
    'SGDRegressor': SGDRegressor(max_iter=1000, tol=1e-3, random_state=42)
}

In [ ]:
# MODEL TRAINING AND EVALUATION
# Training each model and checking its accuracy.

from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    # Calculate R2
    r2 = r2_score(y_test, y_pred)
    
    # Reverse the log transform to calculate REAL dollars
    y_test_usd = np.expm1(y_test)
    y_pred_usd = np.expm1(y_pred)
    
    rmse_usd = np.sqrt(mean_squared_error(y_test_usd, y_pred_usd))
    mae_usd = mean_absolute_error(y_test_usd, y_pred_usd)
    
    results.append({
        'Model': name,
        'R2': r2,
        'RMSE (USD)': rmse_usd,
        'MAE (USD)': mae_usd
    })

results_df = pd.DataFrame(results).sort_values(by='R2', ascending=False)
results_df


In [ ]:
# choosing the best performing model
best_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', models['LinearRegression'])
    
])

best_pipeline.fit(X_train, y_train)


In [ ]:
# Numeric feature names (unchanged)
numeric_features = [
    'log_views',
    'log_subscribers',
    'watch_time_per_view',
    'engagement_rate',
    'video_length_minutes',
    'year',
    'month',
    'dayofweek'
]

# Get encoded categorical feature names
encoded_cat_features = (
    best_pipeline
    .named_steps['preprocessor']
    .named_transformers_['cat']
    .get_feature_names_out(['category', 'device', 'country'])
)

# Combine all feature names
feature_names = numeric_features + list(encoded_cat_features)


In [ ]:
#  FEATURE IMPORTANCE 
# Linear models use coefficients instead of feature importances.

importances = best_pipeline.named_steps['model'].coef_
if importances.ndim > 1:
    importances = importances.flatten()

feature_names = best_pipeline.named_steps['preprocessor'].get_feature_names_out()

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': np.abs(importances)  # Using absolute value to find most impactful
}).sort_values(by='Importance', ascending=False)




In [ ]:
#   EXPLORATORY DATA ANALYSIS (EDA) 
# Visualizing the distribution of the data to find insights.

# Display top features
plt.figure(figsize=(8,4))
sns.barplot(
    x='Importance',
    y='Feature',
    data=importance_df.head(10),
    palette='viridis'
)
plt.title("Top Feature Importances (Linear Coefficients)")
plt.xlabel("Absolute Coefficient Magnitude")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

importance_df.head(10)

In [ ]:
# SAVING THE MODEL 
# Saving our trained model into a .pkl file so the Streamlit App can use it later without retraining.

import joblib

joblib.dump(best_pipeline, "youtube_ad_revenue_model.pkl")


In [ ]:
loaded_model = joblib.load("youtube_ad_revenue_model.pkl")

# test prediction on a few rows
loaded_model.predict(X_test.iloc[:5])


In [ ]:
joblib.dump({
    "numeric_features": numeric_features,
    "categorical_features": categorical_features
}, "feature_config.pkl")
